# 14c THINGS-EEG1 map test over the whole epoch

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 08a and 08b.

Development analysis, not a prespecified test. 08n (P-3) run again with its code unchanged; every score is also
computed over the whole epoch, after onset and per 100 ms window, and the P-3 rule applied to each, descriptively.
The 100 to 200 ms decision must reproduce 08n.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '14c-eeg1epoch', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"sources": ["sub-04", "sub-07", "sub-11", "sub-12", "sub-13", "sub-14", "sub-15", "sub-17", "sub-20", "sub-21", "sub-24", "sub-25", "sub-26", "sub-29", "sub-30", "sub-34", "sub-35", "sub-38", "sub-39", "sub-46", "sub-48", "sub-49"], "targets": ["sub-02", "sub-03", "sub-05", "sub-08", "sub-09", "sub-10", "sub-16", "sub-19", "sub-22", "sub-27", "sub-28", "sub-31", "sub-32", "sub-33", "sub-36", "sub-37", "sub-40", "sub-41", "sub-42", "sub-43", "sub-44", "sub-45", "sub-47", "sub-50"], "budgets": [250, 500], "draws": [13, 14, 15], "window_s": [0.1, 0.2], "map_window_ms": 100, "delta": 0.003547826037718968, "alpha": 0.016666666666666666, "min_targets": 20, "candidate": "h4b", "comparators": ["avg_sources", "gu_window", "linear_converter", "own_ridge", "own_ridge_oc", "generic"], "protocol": "P-3 frozen, follow-up 3 of 3", "feature_sha256": "61d36682db158fc622efe721c7759f70f52868a828aa9a66d395fdff7318ec1f", "k_subsets_sha256": "3f3ede66115c9fefd34e15d0b93a5fc10ef3f12ed32e251a3aab1a79732e01c1"}')
import numpy as np
import pandas as pd

from cross_subject_encoding.data import things_eeg1 as e1
from cross_subject_encoding.data import things_eeg2 as e2
from cross_subject_encoding.eval import eeg1_decision as dec
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.models.baselines import best_source, gu_by_group, window_groups
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.eeg1_pilot import sha256_file, source_file
from cross_subject_encoding.utils.preflight import preflight

feature_path, subsets_path = C.find_artifact("eeg1_paper_kpca1000.npz"), C.find_artifact("eeg1_k_subsets.json")
assert sha256_file(feature_path) == FIXED["feature_sha256"] and sha256_file(subsets_path) == FIXED["k_subsets_sha256"]
subsets = json.loads(subsets_path.read_text())
with np.load(feature_path) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
sources, targets = FIXED["sources"], FIXED["targets"]
opened, y_src = [], {}

def load(person, part):
    path = source_file(Path("/kaggle/input"), person, part, sources + targets)
    opened.append(path.name)
    rel = e2.load_part(path)
    if not np.isfinite(rel["data"]).all():
        raise ValueError(f"{person}/{part}: non-finite EEG")
    return rel["data"], rel["times"]

query = np.concatenate([z_train, z_test])
src_train = np.empty((len(sources), e1.N_TRAIN, 17 * 100), np.float32)
src_test = np.empty((len(sources), e1.N_TEST, 17 * 100), np.float32)
for i, s in enumerate(sources):
    data, times = load(s, "training")
    y_src[s] = data[:, 0]
    pred = pl.ols_paper(z_train, y_src[s].reshape(e1.N_TRAIN, -1), query)
    src_train[i], src_test[i] = pred[:e1.N_TRAIN], pred[e1.N_TRAIN:]
groups = window_groups(times, 17, C.EEG_WINDOW_MS)
map_windows = np.round((times - times[0]) * 1000).astype(int) // FIXED["map_window_ms"]
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
avg_train, avg_test = src_train.mean(0), src_test.mean(0)
# generic map: mean of the maps fitted the same way on each source (others' average of the other 21), per draw and k
generic = {}
for d in FIXED["draws"]:
    for k in FIXED["budgets"]:
        maps = []
        for q, s in enumerate(sources):
            idx_q = np.asarray(subsets["orders"][s][str(d)])[:k]
            rest = [j for j in range(len(sources)) if j != q]
            avg_q = src_train[np.ix_(rest, idx_q)].mean(0).reshape(k, 17, 100)
            maps.append(h4b.fit_map(avg_q, y_src[s][idx_q], map_windows, seed=d)[0])
        generic[(d, k)] = h4b.mean_map(maps)
print("sources and generic maps ready |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
ALPHAS = [10.0 ** e for e in range(-2, 9)]
rows, failed, done = [], {}, []
avg_t3 = avg_test.reshape(-1, 17, 100)
for person in targets:
    t = time.monotonic()
    try:
        train_data, _ = load(person, "training")
        full = train_data[:, 0]
        biological, _ = load(person, "test")
        preds = {"avg_sources|all|all": avg_t3}
        for d in FIXED["draws"]:
            order = np.asarray(subsets["orders"][person][str(d)])
            for k in FIXED["budgets"]:
                idx = order[:k]
                target = full[idx]
                flat = target.reshape(k, -1)
                own_map, _ = h4b.fit_map(avg_train[idx].reshape(k, 17, 100), target, map_windows, seed=d)
                preds[f"h4b|{k}|{d}"] = h4b.apply_map(own_map, avg_t3, map_windows)
                preds[f"generic|{k}|{d}"] = h4b.apply_map(generic[(d, k)], avg_t3, map_windows)
                own = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(z_train[idx], flat)
                own_test = own.predict(z_test)
                oof = np.empty_like(flat)
                for fold in np.array_split(np.random.default_rng(d).permutation(k), C.RIDGE_FOLDS):
                    keep = np.setdiff1d(np.arange(k), fold)
                    oof[fold] = own.refit(z_train[idx[keep]], flat[keep]).predict(z_train[idx[fold]])
                preds[f"own_ridge|{k}|{d}"] = own_test
                preds[f"own_ridge_oc|{k}|{d}"] = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(oof, flat).predict(own_test)
                borrowed = src_train[:, idx]
                best = best_source(list(borrowed), flat)
                preds[f"linear_converter|{k}|{d}"] = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(borrowed[best], flat).predict(src_test[best])
                preds[f"gu_window|{k}|{d}"] = gu_by_group(borrowed, flat, src_test, groups, "pooled")
        preds = {n: np.asarray(p).reshape(e1.N_TEST, 17, 100) for n, p in preds.items()}
        if not all(np.isfinite(p).all() for p in preds.values()):
            raise ValueError("nonfinite prediction")
        scored = paper_correlation(preds, biological, n_iter=100)
        for name, curve in scored["correlation"].items():
            method, k, d = name.split("|")
            rows.append({"person": person, "method": method, "k": k, "draw": d, "r_100_200ms": float(curve[:, window].mean()), "r_whole": float(curve.mean()), "r_post": float(curve[:, times >= 0].mean()), **{f"r_w{int(w):02d}": float(curve[:, map_windows == w].mean()) for w in np.unique(map_windows)}})
        done.append(person)
    except Exception as error:  # P-3: a failed target is excluded and reported, never imputed
        failed[person] = f"{type(error).__name__}: {error}"
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "14c_scores.csv", index=False)
    print(person, f"{time.monotonic() - t:.0f} s", failed.get(person, ""))
frame = pd.DataFrame(rows)
mean = frame.groupby(["person", "method", "k"])["r_100_200ms"].mean()
scores = {}
for (person, method, k), value in mean.items():
    for b in FIXED["budgets"]:
        if k == "all" or k == str(b):
            scores[(person, method, b)] = value
comparators = tuple(FIXED["comparators"])
d = dec.endpoint(scores, targets, FIXED["budgets"], comparators=comparators, candidate=FIXED["candidate"])
decision = dec.decide(d, FIXED["delta"], FIXED["alpha"], FIXED["min_targets"], comparators=comparators)
decision["sign_flip_p"] = {c: dec.sign_flip_p(d[:, j]) for j, c in enumerate(comparators)}
decision["d"] = {p: row.tolist() for p, row in zip(targets, d)}
decision["failed"] = failed
decision["method_means"] = {f"{m}|{b}": float(np.nanmean([scores.get((p, m, b), np.nan) for p in done]))
                            for m in (FIXED["candidate"],) + comparators for b in FIXED["budgets"]}
decision["eeg_files_opened"] = len(opened)

# 14c/14d: the same decision rule on other scoring windows, descriptive (the margin was set for 100-200 ms)
extra = {}
for col in ["r_whole", "r_post"] + sorted(c for c in frame.columns if c.startswith("r_w")):
    m2 = frame.groupby(["person", "method", "k"])[col].mean()
    sc = {}
    for (person, method, k), value in m2.items():
        for b in FIXED["budgets"]:
            if k == "all" or k == str(b):
                sc[(person, method, b)] = value
    d2 = dec.endpoint(sc, targets, FIXED["budgets"], comparators=comparators, candidate=FIXED["candidate"])
    dd = dec.decide(d2, FIXED["delta"], FIXED["alpha"], FIXED["min_targets"], comparators=comparators)
    extra[col] = {"verdict": dd["verdict"], "per_comparator": dd["per_comparator"],
                  "method_means": {f"{m}|{b}": float(np.nanmean([sc.get((p, m, b), np.nan) for p in done]))
                                   for m in (FIXED["candidate"],) + comparators for b in FIXED["budgets"]}}
decision["other_windows_descriptive"] = extra
print("WHOLE EPOCH:", extra["r_whole"]["verdict"], "| POST-ONSET:", extra["r_post"]["verdict"])
(C.METRICS_DIR / "14c_decision.json").write_text(json.dumps(decision, indent=2))
print("VERDICT:", decision["verdict"], "| targets", decision["n_targets"], "| failed", len(failed))
print(json.dumps(decision["per_comparator"], indent=1))